In [ ]:
## import libraries
import sys
import glob
import re

import geopandas as gpd
import cartopy
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
import cmocean.cm as cmo
from matplotlib.gridspec import GridSpec

# cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# import personal modules
# Path to modules
sys.path.append('../modules')
# Import my modules
import global_vars
from plotter import draw_basemap, plot_terrain
from utils import select_months_ds, select_months_df, get_startmon_and_endmon
import customcmaps as ccmap
from load_shapefiles import load_region_shp, load_HUC8
from plot_trajectory_maps import subset_gdf_to_plot

pd.options.display.float_format = "{:,.2f}".format # makes it so pandas tables display only first two decimals

In [ ]:
path_to_data = global_vars.path_to_data
path_to_repo = global_vars.path_to_repo
path_to_out  = '../out/'       # output files (numerical results, intermediate datafiles) -- read & write
path_to_figs = '../figs/'      # figures

In [ ]:
polys = load_HUC8()
regions = load_region_shp(polys)

HUC8_ID_lst = HUC8_ID_lst = [14050001, 14010004, 14020002, 14080101]
idx = (polys.HUC8 == str(HUC8_ID_lst[0])) | (polys.HUC8 == str(HUC8_ID_lst[1])) | (polys.HUC8 == str(HUC8_ID_lst[2])) | (polys.HUC8 == str(HUC8_ID_lst[3]))
subset_polys =  polys[idx]

In [ ]:
ssn = 'NDJFMA'
start_mon, end_mon = get_startmon_and_endmon(ssn)
## load PRISM watershed precip dataset to get list of HUC8s
fname = path_to_data + 'preprocessed/PRISM/PRISM_HUC8_CO_sp.nc'
PRISM = xr.open_dataset(fname)
## add water year to data as coordinate
water_year = (PRISM.date.dt.month >= 10) + PRISM.date.dt.year
PRISM.coords['water_year'] = water_year
HUC8_ID_lst = PRISM.HUC8.values ## get list of HUC8 IDs

## subset to ssn
PRISM = select_months_ds(PRISM, start_mon, end_mon, 'date')
## for each HUC8, what is the total WY precipitation?
PRISM_WY = PRISM.prec.groupby(PRISM.water_year).sum(dim="date").sum('water_year')

## for each HUC8, what is the total WY top-decile precipitation?
PRISM_90 = PRISM.where(PRISM.extreme == 1, drop=True)
PRISM_90WY = PRISM_90.prec.groupby(PRISM_90.water_year).sum(dim="date").sum('water_year')

# Load trajectory GeoJSON data
gdf = gpd.read_file(path_to_repo+"out/trajectories.geojson")
gdf.crs = 'epsg:3857'
gdf = gdf.set_index(pd.to_datetime(gdf['start_date']))

ARDT_lst = ['tARget', 'ar', 'ar_scale']
AR = True
df_lst = []
for j, ARDT in enumerate(ARDT_lst):
    prec_val = []
    for i, HUC8 in enumerate(HUC8_ID_lst):
        tmp = subset_gdf_to_plot(gdf, ARDT, ssn, AR, region=None, basin=None, HUC8=HUC8)
        prec_val.append(tmp.prec.sum())
    
    d = {ARDT: prec_val}
    df = pd.DataFrame(d)
    df_lst.append(df)

df = pd.concat(df_lst, axis=1)
df['HUC8'] = HUC8_ID_lst
df['WY'] = PRISM_WY.values
df['WY90'] = PRISM_90WY.values

for j, ARDT in enumerate(ARDT_lst):
    col_name = ARDT + '_WY_contr'
    df[col_name] = (df[ARDT] / df['WY']) * 100.

    col_name = ARDT + '_WY90_contr'
    df[col_name] = (df[ARDT] / df['WY90']) * 100.

# Perform the join using the 'merge' function
merged_gdf = polys.merge(df, on='HUC8')
merged_gdf

In [ ]:
HUC2_lst = []
for i, HUC8 in enumerate(merged_gdf.HUC8.values):
    first2 = HUC8[:2]
    if (first2 == '14') | (first2 == '13'):
        HUC2_lst.append(True)
    else:
        HUC2_lst.append(False)

In [ ]:
merged_gdf['tARget_WY90_contr'].loc[HUC2_lst].describe()

In [ ]:
merged_gdf['tARget_WY90_contr'].loc[HUC2_lst].median()

In [ ]:
# Set up projection
datacrs = ccrs.PlateCarree()  ## the projection the data is in
mapcrs = ccrs.PlateCarree() ## the projection you want your map displayed in

# Set tick/grid locations
ext1 = [-111., -100., 35.5, 42.5] # extent of CO
dx = np.arange(ext1[0],ext1[1]+2,2)
dy = np.arange(ext1[2]-.5,ext1[3]+1,1)

# list of letters to append to titles
letter_lst = list(map(chr, range(97, 123)))

# Create figure
fig = plt.figure(figsize=(8.5, 11))
fig.dpi = 300
fname = path_to_figs + 'choropleth_map_portrait_{0}'.format(ssn)
fmt = 'png'

nrows = 5
ncols = 2

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 1, 1, 0.05, 0.05], width_ratios = [1, 1], wspace=0.02, hspace=0.02)
## use gs[rows index, columns index] to access grids

# Add color bar axis
cbax = plt.subplot(gs[-1,0]) # colorbar axis
lbl_lst = ['tARget v4', 'Rutz AR', 'AR scale']
row_idx = [0, 1, 2]
b_lons = [False, False, True]

for k, ARDT in enumerate(ARDT_lst):
    print(k, row_idx[k], ARDT)
    ## Add axis for plot
    ax = fig.add_subplot(gs[row_idx[k],0], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext1, xticks=dx, yticks=dy,left_lats=True, right_lats=False, bottom_lons=b_lons[k], mask_ocean=False, coastline=False)

    ## topo with gray shading
    # cs = plot_terrain(ax, ext1)

    # add choropleth watershed fraction
    cbarticks = [10, 20, 30, 40, 50, 60, 70, 80, 90]
    lgnd_kwds={"label": "Fraction of top-decile precipitation (%)", "orientation": "horizontal", "ticks": cbarticks}
    cmap, norm, bnds = ccmap.cmap_segmented(cmo.rain, np.arange(0, 110, 10))
    col_name = ARDT + '_WY90_contr'
    cf = merged_gdf.plot(ax=ax, column=col_name, cmap=cmap, vmin=0, vmax=80, norm=norm, alpha=0.8, legend=True, cax=cbax, legend_kwds=lgnd_kwds)
    polys.plot(ax=ax, edgecolor='grey', color='None', linewidth=0.5, zorder=98)

    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8, zorder=199)

    ## add in four focus watersheds
    subset_polys.plot(ax=ax, edgecolor='white', color='None', linewidth=1., zorder=99)
    # basin.plot(ax=ax, edgecolor='white', color='None', zorder=99)

    ## add in region watershed shape - first time full opacity, second low opacity
    # opac_lst = [1., 0.9]
    # zord_lst = [98, 100]
    # lw_lst = [0.75, 0.3]
    opac_lst = [1.]
    zord_lst = [98]
    lw_lst = [0.75]
    for (opac, zord, lw) in zip(opac_lst, zord_lst, lw_lst):
        regions.plot(ax=ax, edgecolor='k', color='None', linewidth=lw, zorder=zord, alpha=opac)

    # ax.set_title(lbl_lst[k], loc='left', fontsize=14)
    ax.text(-0.16, 0.5, lbl_lst[k], va='bottom', ha='center',
                    rotation='vertical', rotation_mode='anchor', fontsize=13,
                    transform=ax.transAxes)

    ## add a, b, c labels
    titlestring = '({0})'.format(letter_lst[k])
    ax.text(0.029, 0.973, titlestring, ha='left', va='top',
            transform=ax.transAxes, fontsize=12., backgroundcolor='white', zorder=101)

cbax = plt.subplot(gs[-1,1]) # colorbar axis
row_idx = [0, 1, 2]
b_lons = [False, False, True]

for k, ARDT in enumerate(ARDT_lst):
    ## Add axis for plot
    ax = fig.add_subplot(gs[row_idx[k],1], projection=mapcrs)
    ax = draw_basemap(ax, extent=ext1, xticks=dx, yticks=dy,left_lats=False, bottom_lons=b_lons[k], right_lats=False, mask_ocean=False, coastline=False)

    # add choropleth watershed fraction
    cbarticks = [0, 5, 10, 15, 20, 25, 30]
    lgnd_kwds={"label": "Fraction of total precipitation (%)", "orientation": "horizontal", "ticks": cbarticks}
    cmap, norm, bnds = ccmap.cmap_segmented(cmo.rain, np.arange(0, 35, 5))
    col_name = ARDT + '_WY_contr'
    cf = merged_gdf.plot(ax=ax, column=col_name, cmap=cmap, vmin=0, vmax=30, norm=norm, alpha=0.8, legend=True, cax=cbax, legend_kwds=lgnd_kwds)
    polys.plot(ax=ax, edgecolor='grey', color='None', linewidth=0.5, zorder=97)

    ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8, zorder=199)

    ## add in four focus watersheds
    subset_polys.plot(ax=ax, edgecolor='white', color='None', linewidth=1., zorder=99)
    # basin.plot(ax=ax, edgecolor='white', color='None', zorder=99)

    ## add in region watershed shape - first time full opacity, second low opacity
    # opac_lst = [1., 0.9]
    # zord_lst = [98, 100]
    # lw_lst = [0.75, 0.3]
    opac_lst = [1.]
    zord_lst = [98]
    lw_lst = [0.75]
    for (opac, zord, lw) in zip(opac_lst, zord_lst, lw_lst):
        regions.plot(ax=ax, edgecolor='k', color='None', linewidth=lw, zorder=zord, alpha=opac)

    ## add a, b, c labels
    titlestring = '({0})'.format(letter_lst[k+3])
    ax.text(0.029, 0.973, titlestring, ha='left', va='top',
            transform=ax.transAxes, fontsize=12., backgroundcolor='white', zorder=101)
        
fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi)

# Show
plt.show()

In [ ]:
# Set up projection
datacrs = ccrs.PlateCarree()  ## the projection the data is in
mapcrs = ccrs.PlateCarree() ## the projection you want your map displayed in

# Set tick/grid locations
ext1 = [-111., -100., 35.5, 42.5] # extent of CO
dx = np.arange(ext1[0],ext1[1]+2,2)
dy = np.arange(ext1[2]-.5,ext1[3]+1,1)

# list of letters to append to titles
letter_lst = list(map(chr, range(97, 123)))

# Create figure
fig = plt.figure(figsize=(5.5, 5.5))
fig.dpi = 300
fname = path_to_figs + 'choropleth_map_portrait_{0}_tARget'.format(ssn)
fmt = 'png'

nrows = 3
ncols = 1

## Use gridspec to set up a plot with a series of subplots that is
## n-rows by n-columns
gs = GridSpec(nrows, ncols, height_ratios=[1, 0.05, 0.05], width_ratios = [1], wspace=0.02, hspace=0.02)
## use gs[rows index, columns index] to access grids

# Add color bar axis
cbax = plt.subplot(gs[-1,0]) # colorbar axis
ARDT = 'tARget'

## Add axis for plot
ax = fig.add_subplot(gs[0,0], projection=mapcrs)
ax = draw_basemap(ax, extent=ext1, xticks=dx, yticks=dy,left_lats=True, right_lats=False, bottom_lons=True, mask_ocean=False, coastline=False)

## topo with gray shading
# cs = plot_terrain(ax, ext1)

# add choropleth watershed fraction
cbarticks = [10, 20, 30, 40, 50, 60, 70, 80, 90]
lgnd_kwds={"label": "Fraction of top-decile precipitation (%)", "orientation": "horizontal", "ticks": cbarticks}
cmap, norm, bnds = ccmap.cmap_segmented(cmo.rain, np.arange(0, 110, 10))
col_name = ARDT + '_WY90_contr'
cf = merged_gdf.plot(ax=ax, column=col_name, cmap=cmap, vmin=0, vmax=80, norm=norm, alpha=0.8, legend=True, cax=cbax, legend_kwds=lgnd_kwds)
polys.plot(ax=ax, edgecolor='grey', color='None', linewidth=0.5, zorder=98)

ax.add_feature(cfeature.STATES, edgecolor='0.4', linewidth=0.8, zorder=199)

## add in region watershed shape - first time full opacity, second low opacity
# opac_lst = [1., 0.9]
# zord_lst = [98, 100]
# lw_lst = [0.75, 0.3]
opac_lst = [1.]
zord_lst = [98]
lw_lst = [0.75]
for (opac, zord, lw) in zip(opac_lst, zord_lst, lw_lst):
    regions.plot(ax=ax, edgecolor='k', color='None', linewidth=lw, zorder=zord, alpha=opac)

        
fig.savefig('%s.%s' %(fname, fmt), bbox_inches='tight', dpi=fig.dpi)

# Show
plt.show()